# Tri-subject kinship verification: Kaggle runner

This notebook only **runs** the project; all code lives in the Git repo.

**Before running**
1. Settings (right panel): **Accelerator = GPU T4 x2 or P100**, **Internet = On**.
2. **Add Input** → your private dataset `fiw-fids` (or the competition *Recognizing Faces in the Wild*).
3. Edit `REPO_URL` in the next cell.
4. Best: **Save Version → Save & Run All** so it runs in the background and keeps `/kaggle/working` as output.

**Resuming / reusing caches in a later session:** after a saved run, add that notebook version's output as an
input (Add Input → Your Work → this notebook) and set `CACHE_FROM` to its `work` folder.
Finished stages (embeddings, splits) are then skipped.

In [ ]:
# ---- settings -------------------------------------------------------------
REPO_URL = "https://github.com/samikhan05/kinship-verification.git"
BRANCH = "main"
CODE_DATASET = None    # alternative to GitHub: name of a Kaggle dataset holding the repo zip, e.g. "kinship-code"
CACHE_FROM = None      # e.g. "/kaggle/input/kinship-run/work" to reuse an earlier run's caches
WORK = "/kaggle/working/work"
OUT = "/kaggle/working/outputs"

## 1. Get the code

In [ ]:
import glob, os, shutil, subprocess
REPO = "/kaggle/working/kinship-verification"
if CODE_DATASET:
    cfg_file = glob.glob(f"/kaggle/input/{CODE_DATASET}/**/configs/default.yaml", recursive=True)[0]
    shutil.copytree(os.path.dirname(os.path.dirname(cfg_file)), REPO, dirs_exist_ok=True)
elif os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
!git log -1 --oneline 2>/dev/null || echo "(not a git checkout)"

## 2. Install packages
* Kaggle's own torch / CUDA are kept: nothing here reinstalls torch.
* `onnxruntime-gpu` 1.22 is built for CUDA 12 (Kaggle). Newer releases target CUDA 13.
* `insightface` and `facenet-pytorch` are installed with `--no-deps`: insightface would otherwise add the CPU-only
  `onnxruntime` (clashes with the GPU one), and facenet-pytorch 2.6.0 would downgrade torch to < 2.3.
  Their real dependencies (numpy, opencv, scipy, scikit-image, requests, tqdm) are already on Kaggle.

In [ ]:
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu==1.22.0 onnx==1.17.0
!pip install -q insightface==2.1 --no-deps
!pip install -q facenet-pytorch==2.6.0 --no-deps

## 3. Check the GPU (ONNX Runtime must list `CUDAExecutionProvider`)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, onnxruntime as ort, insightface
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())
print("insightface", insightface.__version__)

## 4. Find the data and set paths

In [ ]:
hits = sorted(glob.glob("/kaggle/input/**/mid.csv", recursive=True))
if hits:                                   # Option A: full FIW (family folders with mid.csv)
    DATA_ROOT = os.path.dirname(os.path.dirname(hits[0]))
elif os.path.exists("/kaggle/input/recognizing-faces-in-the-wild/train_relationships.csv"):
    DATA_ROOT = "/kaggle/input/recognizing-faces-in-the-wild"   # Option B
else:
    raise FileNotFoundError("No dataset found - attach fiw-fids (or the competition) with Add Input.")
if CACHE_FROM:
    shutil.copytree(CACHE_FROM, WORK, dirs_exist_ok=True)
    print("Copied caches from", CACHE_FROM)
ARGS = f"--data_root {DATA_ROOT} --work_dir {WORK} --output_dir {OUT}"
print(ARGS)

## 5. Unit tests (CPU, synthetic data, ~10 s)

In [ ]:
!python -m pytest -q

## 6. Step 0: data inspection → `outputs/data_summary.md`

In [ ]:
!python scripts/inspect_data.py {ARGS}

## 7. Face detection + alignment + ArcFace embeddings + gender/age (cached, resumable)
Watch the first log lines: they must say the models run on `CUDAExecutionProvider`.

In [ ]:
!python scripts/extract_embeddings.py {ARGS}

## 8. Family-disjoint splits, triplets, negatives → `work/splits/`, `outputs/triplet_check.jpg`

In [ ]:
!python scripts/prepare.py {ARGS}
from IPython.display import Image, display
display(Image(f"{OUT}/triplet_check.jpg", width=420))

## 9. Baselines E0 (cosine) and E1 (logistic regression / SVM)

In [ ]:
!python scripts/evaluate.py {ARGS} --exp E0 E1

## 10. Report → `outputs/results.md`

In [ ]:
!python scripts/make_report.py {ARGS} > /dev/null
from IPython.display import Markdown
display(Markdown(open(f"{OUT}/results.md", encoding="utf-8").read()))

## 11. Package the outputs for download

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip outputs && ls -lh outputs.zip